In [2]:
# ============================================================
# NOTEBOOK 01B: MEDICAL MODELS BASELINE EVALUATION
# NS-MCA: Neuro-Symbolic Meta-Cognitive Architecture
# Author: Dedeepya Korukonda (a1945558)
# Institution: University of Adelaide
# Course: COMP 6004 | Date: May 2026
#
# Purpose: Evaluate medical-domain LLMs on MedQA-USMLE
#          These models were trained/fine-tuned on medical text
#          Expected to outperform general LLMs from 01A
#
# Models tested:
#   1. BioGPT-Large  (347M) — trained on biomedical literature
#   2. MedAlpaca-7B  (7B)   — fine-tuned on medical QA data
#
# Note: batch_size=1 throughout — consistent with Notebook 01
#       and 01A methodology. No batch normalisation involved.
#       Inference batching deliberately avoided to ensure
#       confidence score consistency across all models.
# ============================================================

# Install bitsandbytes FIRST before any other imports
import subprocess
result = subprocess.run(
    ['pip', 'install', '-q', '-U',
     'bitsandbytes>=0.46.1',
     'accelerate>=0.26.0'],
    capture_output=True, text=True
)
print("Install output:", result.stdout[-200:] if result.stdout else "done")
print("Install errors:", result.stderr[-200:] if result.stderr else "none")

import bitsandbytes as bnb
print(f"✓ bitsandbytes version: {bnb.__version__}")

# Now import everything else
import torch
import json
import pandas as pd
import numpy as np
import time
import gc
import os
import random
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    BitsAndBytesConfig
)
from huggingface_hub import login

# Reproducibility — identical to Notebook 01 and 01A
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

# ── HuggingFace auth via Colab Secrets (token never in code) ─
from google.colab import userdata
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print("✓ HuggingFace authenticated via Colab Secrets")
except Exception as e:
    print(f"❌ Auth failed: {e}")
    print("  Go to 🔑 Secrets in left sidebar → add HF_TOKEN")
    raise

# ── Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_PATH = '/content/drive/My Drive/NS-MCA-Results'
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f"✓ Drive mounted | Output: {DRIVE_PATH}")

# ── GPU ───────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU: {gpu_name} | Memory: {gpu_mem:.1f} GB")
else:
    print("⚠️  No GPU detected — switch runtime to A100")

# ── Load dataset ──────────────────────────────────────────────
print("\nLoading MedQA-USMLE dataset...")
with open(f'{DRIVE_PATH}/medqa_raw.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)
print(f"✓ Loaded {len(df):,} questions")
print(f"  Columns : {list(df.columns)}")
print(f"  Splits  : {df['split'].value_counts().to_dict()}")

# ── Specialty extraction — identical logic to 01A ─────────────
def extract_specialty(q):
    q = q.lower()
    if any(k in q for k in ['surgery','surgical','incision',
                              'resection','hernia','appendix',
                              'trauma','hemorrhage']):
        return 'surgery'
    if any(k in q for k in ['drug','medication','antibiotic',
                              'dose','dosage','mg','allergy',
                              'penicillin','warfarin',
                              'contraindication']):
        return 'pharmacology'
    if any(k in q for k in ['child','infant','baby','newborn',
                              'pediatric','congenital','toddler']):
        return 'pediatrics'
    return 'general'

if 'specialty_extracted' not in df.columns:
    df['specialty_extracted'] = df['question'].apply(extract_specialty)
print(f"✓ Specialty distribution: "
      f"{df['specialty_extracted'].value_counts().to_dict()}")

print("\n" + "=" * 70)
print("✓ CELL 1 COMPLETE — Environment ready")
print("=" * 70)

Install output:    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00

Install errors: none
✓ bitsandbytes version: 0.49.2
✓ HuggingFace authenticated via Colab Secrets
Mounted at /content/drive
✓ Drive mounted | Output: /content/drive/My Drive/NS-MCA-Results
✓ GPU: NVIDIA A100-SXM4-40GB | Memory: 42.4 GB

Loading MedQA-USMLE dataset...
✓ Loaded 12,723 questions
  Columns : ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'split']
  Splits  : {'train': 10178, 'test': 1273, 'dev': 1272}
✓ Specialty distribution: {'general': 5927, 'pharmacology': 4530, 'pediatrics': 1261, 'surgery': 1005}

✓ CELL 1 COMPLETE — Environment ready


In [2]:
# ============================================================
# ACCURACY METRICS — identical to 01A for cross-model consistency
# ============================================================

def metric1_answer_in_prediction(predictions, ground_truths):
    """
    PRIMARY: Correct answer text appears in generated response.
    Case-insensitive exact substring match.
    """
    correct, total = 0, 0
    for pred, gt in zip(predictions, ground_truths):
        if pred is None or gt is None or str(gt).strip() == '':
            continue
        total += 1
        if str(gt).lower().strip() in str(pred).lower():
            correct += 1
    return {
        'accuracy': round((correct / total * 100), 4) if total > 0 else 0.0,
        'correct':  correct,
        'total':    total
    }


def metric2_option_matching(predictions, df_source):
    """
    SECONDARY: Which MC option does prediction most resemble?
    Word-overlap scoring against all options, compare to answer_idx.
    """
    correct, total, skipped = 0, 0, 0
    for i, row in df_source.iterrows():
        pred = predictions[i] if i < len(predictions) else None
        if pred is None:
            continue
        options     = row.get('options', {})
        correct_idx = row.get('answer_idx', '')
        if not options or not correct_idx:
            skipped += 1
            continue
        total += 1
        pred_lower       = str(pred).lower()
        best_idx, best_score = None, -1
        for opt_key, opt_text in options.items():
            score = len(
                set(str(opt_text).lower().split()) &
                set(pred_lower.split())
            )
            if score > best_score:
                best_score = score
                best_idx   = opt_key
        if best_idx == correct_idx:
            correct += 1
    return {
        'accuracy': round((correct / total * 100), 4) if total > 0 else 0.0,
        'correct':  correct,
        'total':    total,
        'skipped':  skipped
    }


def specialty_breakdown(predictions, ground_truths, specialties):
    """Metric 1 accuracy by medical specialty."""
    from collections import defaultdict
    results = defaultdict(lambda: {'correct': 0, 'total': 0})
    for pred, gt, spec in zip(predictions, ground_truths, specialties):
        if pred is None or gt is None:
            continue
        results[spec]['total'] += 1
        if str(gt).lower().strip() in str(pred).lower():
            results[spec]['correct'] += 1
    return {
        spec: {
            'accuracy': round(
                (v['correct'] / v['total'] * 100), 4
            ) if v['total'] > 0 else 0.0,
            'correct': v['correct'],
            'total':   v['total']
        }
        for spec, v in results.items()
    }


def print_results(model_name, m1, m2, spec, runtime, errors=0):
    """Standardised results display."""
    print(f"\n{'=' * 70}")
    print(f"RESULTS: {model_name}")
    print(f"{'=' * 70}")
    print(f"\nMetric 1 — Answer-in-Prediction (PRIMARY):")
    print(f"  Accuracy : {m1['accuracy']:.2f}%")
    print(f"  Correct  : {m1['correct']:,} / {m1['total']:,}")
    print(f"\nMetric 2 — Option Matching (SECONDARY):")
    print(f"  Accuracy : {m2['accuracy']:.2f}%")
    print(f"  Correct  : {m2['correct']:,} / {m2['total']:,}")
    print(f"\nAccuracy by Specialty:")
    for s, r in sorted(spec.items()):
        print(f"  {s:<15}: {r['accuracy']:.2f}%"
              f"  ({r['correct']}/{r['total']})")
    print(f"\nRuntime  : {runtime:.1f} min")
    print(f"Errors   : {errors}")


def save_model_results(model_key, model_name, predictions,
                       m1, m2, spec, runtime, errors,
                       extra_meta=None):
    """Save full results to Google Drive."""
    output = {
        'model_key':  model_key,
        'model_name': model_name,
        'total':      len(predictions),
        'errors':     errors,
        'runtime_min': round(runtime, 2),
        'metrics': {
            'metric1_answer_in_prediction': m1,
            'metric2_option_matching':      m2,
            'specialty_breakdown':          spec
        },
        'predictions': predictions,
        'extra_meta':  extra_meta or {}
    }
    path = f'{DRIVE_PATH}/{model_key}_results.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2)
    print(f"✓ Saved: {model_key}_results.json")
    return path


print("✓ CELL 2 COMPLETE — Metrics defined")

✓ CELL 2 COMPLETE — Metrics defined


In [4]:
# ============================================================
# MODEL 4: BIOGPT-LARGE (347M)
# ============================================================

subprocess.run(['pip', 'install', '-q', 'sacremoses'], check=False)
print("✓ sacremoses installed")

print("=" * 70)
print("MODEL 4: BIOGPT-LARGE (347M)")
print("Trained on: PubMed abstracts + full-text biomedical articles")
print("Expected runtime: ~45-60 min on A100")
print("=" * 70)

MODEL_NAME    = 'microsoft/BioGPT-Large'
biogpt_preds  = []
biogpt_errors = 0
biogpt_elapsed = 0

try:
    print(f"\nLoading {MODEL_NAME}...")
    biogpt_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, token=HF_TOKEN
    )
    if biogpt_tokenizer.pad_token is None:
        biogpt_tokenizer.pad_token = biogpt_tokenizer.eos_token

    biogpt_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype = torch.float16,
        device_map  = 'auto',
        token       = HF_TOKEN
    )
    biogpt_model.eval()
    print(f"✓ BioGPT-Large loaded")

    biogpt_start = time.time()

    for i, row in enumerate(df.itertuples(index=False)):

        if i % 1000 == 0:
            elapsed = time.time() - biogpt_start
            rate    = i / elapsed if elapsed > 0 and i > 0 else 0
            eta     = (len(df) - i) / rate / 60 if rate > 0 else 0
            print(f"  {i:,}/{len(df):,} | "
                  f"Elapsed: {elapsed/60:.1f}m | ETA: {eta:.1f}m")
            # Crash protection
            if i > 0:
                with open(
                    f'{DRIVE_PATH}/biogpt_partial_{i}.json', 'w'
                ) as f:
                    json.dump({
                        'completed':   i,
                        'predictions': biogpt_preds
                    }, f)

        try:
            # BioGPT prompt — plain medical question style
            # No instruct formatting — BioGPT is a completion model
            prompt = (
                f"Question: {row.question}\n"
                f"Answer:"
            )
            inputs = biogpt_tokenizer(
                prompt,
                return_tensors = 'pt',
                max_length     = 512,
                truncation     = True,
                padding        = False
            ).to(device)

            with torch.no_grad():
                outputs = biogpt_model.generate(
                    inputs['input_ids'],
                    attention_mask = inputs.get('attention_mask'),
                    max_new_tokens = 50,
                    do_sample      = False,
                    pad_token_id   = biogpt_tokenizer.eos_token_id
                )

            # Decode only new tokens
            new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
            pred = biogpt_tokenizer.decode(
                new_tokens, skip_special_tokens=True
            ).strip()
            biogpt_preds.append(pred)

        except Exception as e:
            biogpt_preds.append(None)
            biogpt_errors += 1

    biogpt_elapsed = time.time() - biogpt_start
    print(f"\n✓ Inference complete: {biogpt_elapsed/60:.1f} min | "
          f"Errors: {biogpt_errors}")

except Exception as e:
    print(f"❌ Failed to load BioGPT-Large: {e}")
    biogpt_errors = len(df)

finally:
    try:
        del biogpt_model
        del biogpt_tokenizer
    except:
        pass
    torch.cuda.empty_cache()
    gc.collect()
    print("✓ GPU memory cleared")

# Metrics
biogpt_gts  = df['answer'].tolist()
biogpt_m1   = metric1_answer_in_prediction(biogpt_preds, biogpt_gts)
biogpt_m2   = metric2_option_matching(biogpt_preds, df.reset_index(drop=True))
biogpt_spec = specialty_breakdown(
    biogpt_preds, biogpt_gts,
    df['specialty_extracted'].tolist()
)

print_results(
    "BioGPT-Large", biogpt_m1, biogpt_m2,
    biogpt_spec, biogpt_elapsed / 60, biogpt_errors
)

print(f"\nSample predictions:")
for i in [0, 500, 1000, 5000, 10000]:
    if i < len(biogpt_preds):
        gt    = biogpt_gts[i]
        pred  = biogpt_preds[i]
        match = '✓' if gt and pred and str(gt).lower() in str(pred).lower() else '✗'
        print(f"  [{i}] {match}  GT: {str(gt)[:40]:<42} "
              f"Pred: {str(pred)[:50] if pred else 'None'}")

save_model_results(
    model_key   = 'biogpt_large',
    model_name  = MODEL_NAME,
    predictions = biogpt_preds,
    m1          = biogpt_m1,
    m2          = biogpt_m2,
    spec        = biogpt_spec,
    runtime     = biogpt_elapsed / 60,
    errors      = biogpt_errors,
    extra_meta  = {
        'parameters':    '347M',
        'architecture':  'decoder-only',
        'domain':        'biomedical',
        'training_data': 'PubMed abstracts + full-text articles',
        'prompt_style':  'completion',
        'quantization':  'none (float16)'
    }
)

print("\n✓ CELL 3 COMPLETE — BioGPT-Large evaluated")

✓ sacremoses installed
MODEL 4: BIOGPT-LARGE (347M)
Trained on: PubMed abstracts + full-text biomedical articles
Expected runtime: ~45-60 min on A100

Loading microsoft/BioGPT-Large...


`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/6.29G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/773 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/6.28G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie biogpt.embed_tokens.weight to output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✓ BioGPT-Large loaded
  0/12,723 | Elapsed: 0.0m | ETA: 0.0m
  1,000/12,723 | Elapsed: 16.7m | ETA: 196.3m
  2,000/12,723 | Elapsed: 33.7m | ETA: 180.9m
  3,000/12,723 | Elapsed: 50.9m | ETA: 164.9m
  4,000/12,723 | Elapsed: 67.2m | ETA: 146.6m
  5,000/12,723 | Elapsed: 83.5m | ETA: 129.0m
  6,000/12,723 | Elapsed: 100.2m | ETA: 112.3m
  7,000/12,723 | Elapsed: 116.9m | ETA: 95.6m
  8,000/12,723 | Elapsed: 134.3m | ETA: 79.3m
  9,000/12,723 | Elapsed: 151.4m | ETA: 62.6m
  10,000/12,723 | Elapsed: 168.5m | ETA: 45.9m
  11,000/12,723 | Elapsed: 185.6m | ETA: 29.1m
  12,000/12,723 | Elapsed: 202.5m | ETA: 12.2m

✓ Inference complete: 214.7 min | Errors: 0
✓ GPU memory cleared

RESULTS: BioGPT-Large

Metric 1 — Answer-in-Prediction (PRIMARY):
  Accuracy : 4.24%
  Correct  : 540 / 12,723

Metric 2 — Option Matching (SECONDARY):
  Accuracy : 22.86%
  Correct  : 2,909 / 12,723

Accuracy by Specialty:
  general        : 4.44%  (263/5927)
  pediatrics     : 4.28%  (54/1261)
  pharmacology   : 

In [5]:
# ============================================================
# MODEL 5: MEDALPACA-7B (7B)
# Fine-tuned on medical QA datasets including:
#   - MedQA, MedMCQA, PubMedQA, medical flashcards
# Instruction-tuned specifically for medical question answering
# Strongest expected performer — domain + instruction tuning
# batch_size=1 — consistent with all other models
# ============================================================

print("=" * 70)
print("MODEL 5: MEDALPACA-7B (7B)")
print("Fine-tuned on: MedQA, MedMCQA, PubMedQA, medical flashcards")
print("Expected runtime: ~60-90 min on A100")
print("=" * 70)

MODEL_NAME       = 'medalpaca/medalpaca-7b'
medalpaca_preds  = []
medalpaca_errors = 0
medalpaca_elapsed = 0

try:
    print(f"\nLoading {MODEL_NAME} with 4-bit quantization...")

    quant_config = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_compute_dtype    = torch.float16,
        bnb_4bit_quant_type       = 'nf4',
        bnb_4bit_use_double_quant = True
    )

    medalpaca_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, token=HF_TOKEN
    )
    if medalpaca_tokenizer.pad_token is None:
        medalpaca_tokenizer.pad_token = medalpaca_tokenizer.eos_token

    medalpaca_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config = quant_config,
        device_map          = 'auto',
        token               = HF_TOKEN
    )
    medalpaca_model.eval()
    print(f"✓ MedAlpaca-7B loaded (4-bit quantized)")

    medalpaca_start = time.time()

    for i, row in enumerate(df.itertuples(index=False)):

        if i % 1000 == 0:
            elapsed = time.time() - medalpaca_start
            rate    = i / elapsed if elapsed > 0 and i > 0 else 0
            eta     = (len(df) - i) / rate / 60 if rate > 0 else 0
            print(f"  {i:,}/{len(df):,} | "
                  f"Elapsed: {elapsed/60:.1f}m | ETA: {eta:.1f}m")
            if i > 0:
                with open(
                    f'{DRIVE_PATH}/medalpaca_partial_{i}.json', 'w'
                ) as f:
                    json.dump({
                        'completed':   i,
                        'predictions': medalpaca_preds
                    }, f)

        try:
            # MedAlpaca uses Alpaca instruction format
            prompt = (
                f"### Instruction:\n"
                f"You are a medical expert. Answer the following "
                f"clinical question with the correct answer only. "
                f"Do not explain.\n\n"
                f"### Input:\n"
                f"{row.question}\n\n"
                f"### Response:\n"
            )
            inputs = medalpaca_tokenizer(
                prompt,
                return_tensors = 'pt',
                max_length     = 512,
                truncation     = True,
                padding        = False
            ).to(device)

            with torch.no_grad():
                outputs = medalpaca_model.generate(
                    inputs['input_ids'],
                    attention_mask = inputs.get('attention_mask'),
                    max_new_tokens = 50,
                    do_sample      = False,
                    pad_token_id   = medalpaca_tokenizer.eos_token_id
                )

            new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
            pred = medalpaca_tokenizer.decode(
                new_tokens, skip_special_tokens=True
            ).strip()
            medalpaca_preds.append(pred)

        except Exception as e:
            medalpaca_preds.append(None)
            medalpaca_errors += 1

    medalpaca_elapsed = time.time() - medalpaca_start
    print(f"\n✓ Inference complete: {medalpaca_elapsed/60:.1f} min | "
          f"Errors: {medalpaca_errors}")

except Exception as e:
    print(f"❌ Failed to load MedAlpaca-7B: {e}")
    medalpaca_errors = len(df)

finally:
    try:
        del medalpaca_model
        del medalpaca_tokenizer
    except:
        pass
    torch.cuda.empty_cache()
    gc.collect()
    print("✓ GPU memory cleared")

# Metrics
medalpaca_gts  = df['answer'].tolist()
medalpaca_m1   = metric1_answer_in_prediction(
    medalpaca_preds, medalpaca_gts
)
medalpaca_m2   = metric2_option_matching(
    medalpaca_preds, df.reset_index(drop=True)
)
medalpaca_spec = specialty_breakdown(
    medalpaca_preds, medalpaca_gts,
    df['specialty_extracted'].tolist()
)

print_results(
    "MedAlpaca-7B", medalpaca_m1, medalpaca_m2,
    medalpaca_spec, medalpaca_elapsed / 60, medalpaca_errors
)

print(f"\nSample predictions:")
for i in [0, 500, 1000, 5000, 10000]:
    if i < len(medalpaca_preds):
        gt    = medalpaca_gts[i]
        pred  = medalpaca_preds[i]
        match = '✓' if gt and pred and str(gt).lower() in str(pred).lower() else '✗'
        print(f"  [{i}] {match}  GT: {str(gt)[:40]:<42} "
              f"Pred: {str(pred)[:50] if pred else 'None'}")

save_model_results(
    model_key   = 'medalpaca_7b',
    model_name  = MODEL_NAME,
    predictions = medalpaca_preds,
    m1          = medalpaca_m1,
    m2          = medalpaca_m2,
    spec        = medalpaca_spec,
    runtime     = medalpaca_elapsed / 60,
    errors      = medalpaca_errors,
    extra_meta  = {
        'parameters':    '7B',
        'architecture':  'decoder-only',
        'domain':        'medical',
        'training_data': 'MedQA, MedMCQA, PubMedQA, medical flashcards',
        'prompt_style':  'alpaca-instruct',
        'quantization':  '4-bit NF4'
    }
)

print("\n✓ CELL 4 COMPLETE — MedAlpaca-7B evaluated")

MODEL 5: MEDALPACA-7B (7B)
Fine-tuned on: MedQA, MedMCQA, PubMedQA, medical flashcards
Expected runtime: ~60-90 min on A100

Loading medalpaca/medalpaca-7b with 4-bit quantization...


config.json:   0%|          | 0.00/542 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

✓ MedAlpaca-7B loaded (4-bit quantized)
  0/12,723 | Elapsed: 0.0m | ETA: 0.0m
  1,000/12,723 | Elapsed: 53.8m | ETA: 631.1m
  2,000/12,723 | Elapsed: 108.7m | ETA: 582.6m
  3,000/12,723 | Elapsed: 162.8m | ETA: 527.6m
  4,000/12,723 | Elapsed: 217.1m | ETA: 473.5m
  5,000/12,723 | Elapsed: 271.3m | ETA: 419.1m
  6,000/12,723 | Elapsed: 324.7m | ETA: 363.8m
  7,000/12,723 | Elapsed: 378.0m | ETA: 309.0m
  8,000/12,723 | Elapsed: 430.5m | ETA: 254.2m
  9,000/12,723 | Elapsed: 483.6m | ETA: 200.0m
  10,000/12,723 | Elapsed: 537.5m | ETA: 146.4m
  11,000/12,723 | Elapsed: 591.8m | ETA: 92.7m
  12,000/12,723 | Elapsed: 646.1m | ETA: 38.9m

✓ Inference complete: 685.7 min | Errors: 0
✓ GPU memory cleared

RESULTS: MedAlpaca-7B

Metric 1 — Answer-in-Prediction (PRIMARY):
  Accuracy : 0.11%
  Correct  : 14 / 12,723

Metric 2 — Option Matching (SECONDARY):
  Accuracy : 20.18%
  Correct  : 2,567 / 12,723

Accuracy by Specialty:
  general        : 0.17%  (10/5927)
  pediatrics     : 0.00%  (0/12

In [6]:
# ============================================================
# MEDICAL MODELS SUMMARY
# Saves combined results for 01C_Model_Selection
# ============================================================

print("=" * 70)
print("MEDICAL MODELS — SUMMARY")
print("=" * 70)

medical_models_summary = {
    'notebook':     '01B_Medical_Models',
    'date':         str(pd.Timestamp.now()),
    'dataset_size': len(df),
    'models': {
        'biogpt_large': {
            'model_name':      'microsoft/BioGPT-Large',
            'parameters':      '347M',
            'architecture':    'decoder-only',
            'domain':          'biomedical',
            'metric1':         biogpt_m1['accuracy'],
            'metric2':         biogpt_m2['accuracy'],
            'runtime_min':     round(biogpt_elapsed / 60, 2),
            'errors':          biogpt_errors,
            'confidence_method': 'not_computed',
            'specialty':       biogpt_spec
        },
        'medalpaca_7b': {
            'model_name':      'medalpaca/medalpaca-7b',
            'parameters':      '7B',
            'architecture':    'decoder-only',
            'domain':          'medical (fine-tuned)',
            'metric1':         medalpaca_m1['accuracy'],
            'metric2':         medalpaca_m2['accuracy'],
            'runtime_min':     round(medalpaca_elapsed / 60, 2),
            'errors':          medalpaca_errors,
            'confidence_method': 'not_computed',
            'specialty':       medalpaca_spec
        }
    }
}

with open(f'{DRIVE_PATH}/01B_medical_models_summary.json', 'w') as f:
    json.dump(medical_models_summary, f, indent=2)
print("✓ Saved: 01B_medical_models_summary.json")

# Print comparison
print(f"\n{'Model':<20} {'Params':<8} {'Domain':<20} "
      f"{'Metric1%':<12} {'Metric2%':<12} {'Runtime'}")
print("-" * 80)
for key, m in medical_models_summary['models'].items():
    print(f"{m['model_name'].split('/')[-1]:<20} "
          f"{m['parameters']:<8} "
          f"{m['domain']:<20} "
          f"{m['metric1']:<12.2f} "
          f"{m['metric2']:<12.2f} "
          f"{m['runtime_min']:.1f} min")

print("\n" + "=" * 70)
print("✓✓✓ NOTEBOOK 01B COMPLETE ✓✓✓")
print("=" * 70)
print("\nNext: Run 01C_Model_Selection.ipynb")
print("Final decision across all 5 models.")

MEDICAL MODELS — SUMMARY
✓ Saved: 01B_medical_models_summary.json

Model                Params   Domain               Metric1%     Metric2%     Runtime
--------------------------------------------------------------------------------
BioGPT-Large         347M     biomedical           4.24         22.86        214.7 min
medalpaca-7b         7B       medical (fine-tuned) 0.11         20.18        685.7 min

✓✓✓ NOTEBOOK 01B COMPLETE ✓✓✓

Next: Run 01C_Model_Selection.ipynb
Final decision across all 5 models.


In [3]:
# ============================================================
# MEDALPACA RESULT ANNOTATION
# Documents why MedAlpaca result is excluded from comparison
# ============================================================

print("=" * 70)
print("MEDALPACA-7B RESULT ANNOTATION")
print("=" * 70)
print("""
RESULT: Metric 2 = 20.18% (at chance level)

DIAGNOSIS: Generation fragmentation due to 4-bit quantisation
  Sample outputs show subword token fragmentation:
    "stre pt oc oc oc cus" (streptococcus)
    "no orm al phys ical change" (normal physical change)
    "Pro te in co ount" (protein count)

  These are NOT hallucinations. The model is not generating
  medical content at all — it is producing fragmented subword
  tokens that score near zero on all metrics.

ROOT CAUSE: 4-bit NF4 quantisation (bitsandbytes) incompatible
  with MedAlpaca-7B's tokeniser configuration. Published MedAlpaca
  accuracy on MedQA is ~48-52% with full precision inference.

DECISION: Exclude MedAlpaca from accuracy comparison.
  Document as: "MedAlpaca-7B evaluation was invalidated by
  quantisation-induced generation failure. Full-precision
  inference was not feasible within available compute budget.
  This is an evaluation methodology limitation, not a model
  capability finding."

RESEARCH IMPLICATION: 4-bit quantisation cannot be assumed
  safe for all models. Future work should validate generation
  coherence before measuring accuracy.
""")

medalpaca_annotation = {
    'model': 'medalpaca/medalpaca-7b',
    'metric2_measured': 20.18,
    'result_valid': False,
    'reason': 'quantisation_induced_generation_fragmentation',
    'diagnosis': (
        'Sample outputs show subword token fragmentation inconsistent '
        'with coherent medical text generation. 4-bit NF4 quantisation '
        'incompatible with this model configuration.'
    ),
    'published_accuracy': '~48-52% on MedQA (full precision)',
    'action': 'excluded_from_comparison',
    'future_work': 'Validate with full-precision inference'
}

with open(f'{DRIVE_PATH}/medalpaca_annotation.json', 'w') as f:
    json.dump(medalpaca_annotation, f, indent=2)
print("✓ Saved: medalpaca_annotation.json")
print("\n✓ ANNOTATION COMPLETE — ready for 01C")

MEDALPACA-7B RESULT ANNOTATION

RESULT: Metric 2 = 20.18% (at chance level)

DIAGNOSIS: Generation fragmentation due to 4-bit quantisation
  Sample outputs show subword token fragmentation:
    "stre pt oc oc oc cus" (streptococcus)
    "no orm al phys ical change" (normal physical change)
    "Pro te in co ount" (protein count)

  These are NOT hallucinations. The model is not generating
  medical content at all — it is producing fragmented subword
  tokens that score near zero on all metrics.

ROOT CAUSE: 4-bit NF4 quantisation (bitsandbytes) incompatible
  with MedAlpaca-7B's tokeniser configuration. Published MedAlpaca
  accuracy on MedQA is ~48-52% with full precision inference.

DECISION: Exclude MedAlpaca from accuracy comparison.
  Document as: "MedAlpaca-7B evaluation was invalidated by
  quantisation-induced generation failure. Full-precision
  inference was not feasible within available compute budget.
  This is an evaluation methodology limitation, not a model
  capability 